<a href="https://colab.research.google.com/github/AWADKILLERB/jupyter/blob/master/Microscopic_Valley_Toolkit_Part7_VQD_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Microscopic Valley Toolkit — Part 7: Reaching the Conduction Band via Variational Quantum Deflation

**Title:** Microscopic Theory of Valley Splitting in Silicon
**Author:** Awad Mohamed
**Supervisor (Prospective):** Dr. Mark Friesen (University of Wisconsin–Madison)
**Project Status:** Stage VII — Climbing the Spectrum to the Valley-Relevant Band

---

## Purpose of this notebook

Part 6 demonstrated that the verified atomic Hamiltonian, mapped onto a
ten-qubit circuit, correctly reproduces the lowest eigenvalue when solved with
the variational quantum eigensolver on a local simulator. That lowest state is
the deepest valence band, not the conduction-band state that valley splitting
actually depends on.

This notebook extends the calculation using variational quantum deflation: the
same optimization is repeated four more times, each time adding a penalty term
that discourages the algorithm from returning to any state already found,
allowing it to climb the spectrum one level at a time until it reaches the
fifth eigenvalue, the conduction-band state used throughout every classical
stage of this project.

The calculation is carried out directly at the physically relevant point in
reciprocal space, the self-derived valley wavevector of Part 3, rather than at
the center of the Brillouin zone.


In [ ]:
!pip install qiskit-nature

import numpy as np
import scipy.linalg as la
import warnings, time
warnings.filterwarnings("ignore")

from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit.circuit.library import ExcitationPreserving
from qiskit.circuit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_algorithms.optimizers import COBYLA

np.set_printoptions(precision=6, suppress=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.9 MB/s eta 0:00:00


## Chapter 1 — The Hamiltonian at the Valley Wavevector

The same verified sp3s* Hamiltonian is evaluated at the self-derived valley
wavevector found in Part 3, rather than at the center of the Brillouin zone
used in Part 6. This is the wavevector actually relevant to valley splitting.


In [ ]:

params_sp3s = {
    "Es": -4.2000, "Ep": 1.7150, "Ess": 6.6850,
    "Vss_sigma": -8.3000, "Vsp_sigma": 5.7292, "Vssp_sigma": 5.3749,
    "a": 5.4310000000,
}
Vxx, Vxy = 1.7150, 4.5750
params_sp3s["Vpp_pi"] = Vxx - Vxy
params_sp3s["Vpp_sigma"] = Vxx + 2*Vxy
a = params_sp3s["a"]

def phase_factors(kx, ky, kz, a):
    kxp, kyp, kzp = kx*a/4.0, ky*a/4.0, kz*a/4.0
    g0 = np.cos(kxp)*np.cos(kyp)*np.cos(kzp) - np.sin(kxp)*np.sin(kyp)*np.sin(kzp)*1j
    g1 = -np.cos(kxp)*np.sin(kyp)*np.sin(kzp) + np.sin(kxp)*np.cos(kyp)*np.cos(kzp)*1j
    g2 = -np.sin(kxp)*np.cos(kyp)*np.sin(kzp) + np.cos(kxp)*np.sin(kyp)*np.cos(kzp)*1j
    g3 = -np.sin(kxp)*np.sin(kyp)*np.cos(kzp) + np.cos(kxp)*np.cos(kyp)*np.sin(kzp)*1j
    return g0, g1, g2, g3

def H_k_sp3s(kx, ky, kz, p):
    Es, Ep, Ess = p["Es"], p["Ep"], p["Ess"]
    Vss, Vsp, Vssp = p["Vss_sigma"], p["Vsp_sigma"], p["Vssp_sigma"]
    Vpp_s, Vpp_p = p["Vpp_sigma"], p["Vpp_pi"]
    a = p["a"]
    Vxx_ = (Vpp_s + 2*Vpp_p)/3.0
    Vxy_ = (Vpp_s - Vpp_p)/3.0
    g0, g1, g2, g3 = phase_factors(kx, ky, kz, a)
    HAA = np.diag([Es, Ep, Ep, Ep, Ess])
    HBB = np.diag([Es, Ep, Ep, Ep, Ess])
    HAB = np.array([
        [Vss*g0,   Vsp*g1,   Vsp*g2,   Vsp*g3,   0],
        [-Vsp*g1,  Vxx_*g0,  Vxy_*g3,  Vxy_*g2,  -Vssp*g1],
        [-Vsp*g2,  Vxy_*g3,  Vxx_*g0,  Vxy_*g1,  -Vssp*g2],
        [-Vsp*g3,  Vxy_*g2,  Vxy_*g1,  Vxx_*g0,  -Vssp*g3],
        [0,        Vssp*g1,  Vssp*g2,  Vssp*g3,  0]
    ], dtype=complex)
    return np.block([[HAA, HAB], [HAB.conj().T, HBB]])

X_abs = 2*np.pi/a
k0 = 0.7307 * X_abs
H_valley = H_k_sp3s(0.0, 0.0, k0, params_sp3s)
M = H_valley.shape[0]
evals_classical = np.sort(la.eigvalsh(H_valley))

print(f"Self-derived valley wavevector: {k0:.6f} inverse angstrom")
print(f"Classical eigenvalues at this wavevector:\n{np.round(evals_classical, 6)}")
print(f"\nTarget conduction-band eigenvalue (index 4): {evals_classical[4]:.6f} eV")
print(f"\nNote: the second and third eigenvalues are exactly degenerate,")
print(f"{evals_classical[2]:.6f} = {evals_classical[3]:.6f}, which is expected to make")
print(f"deflation past this pair numerically more demanding than the other levels.")


Self-derived valley wavevector: 0.845355 inverse angstrom
Classical eigenvalues at this wavevector:
[-10.074782  -6.057785  -2.515727  -2.515727   1.171339   2.72046
   5.945727   5.945727  10.240546  10.400222]

Target conduction-band eigenvalue (index 4): 1.171339 eV

Note: the second and third eigenvalues are exactly degenerate,
-2.515727 = -2.515727, which is expected to make
deflation past this pair numerically more demanding than the other levels.


## Chapter 2 — Mapping to Qubits and Building the Circuit

The same information-preserving Jordan-Wigner mapping and particle-conserving
ansatz used in Part 6 are reused here without modification.


In [ ]:

fermionic_terms = {
    f"+_{p} -_{q}": H_valley[p, q]
    for p in range(M) for q in range(M)
    if abs(H_valley[p, q]) > 1e-12
}
fermionic_op = FermionicOp(fermionic_terms, num_spin_orbitals=M)
qubit_op = JordanWignerMapper().map(fermionic_op)

reference = QuantumCircuit(M)
reference.x(0)
ansatz = ExcitationPreserving(M, reps=2, insert_barriers=False)
full_circuit = reference.compose(ansatz)

print(f"Qubits: {qubit_op.num_qubits}, Pauli terms: {len(qubit_op)}")
print(f"Variational parameters per state: {ansatz.num_parameters}")


Qubits: 10, Pauli terms: 31
Variational parameters per state: 120


## Chapter 3 — Variational Quantum Deflation

Each level is found by minimizing the expectation value of the Hamiltonian plus
a penalty proportional to the squared overlap with every state already found.
A penalty strength and number of random restarts were chosen, after some
experimentation, to reliably separate the exactly degenerate pair at levels two
and three; the same settings are used uniformly for every level here for
consistency and reproducibility.


In [ ]:

beta = 12.0
n_attempts = 3
maxiter = 900

def cost_function(params, prev_states):
    bound = full_circuit.assign_parameters(params)
    sv = Statevector(bound)
    energy = np.real(sv.expectation_value(qubit_op))
    penalty = sum(beta * np.abs(prev.inner(sv))**2 for prev in prev_states)
    return energy + penalty

found_states = []
found_energies = []

t_start = time.time()
for level in range(5):
    np.random.seed(101 + level)
    best = None
    for attempt in range(n_attempts):
        x0 = np.random.uniform(-0.5, 0.5, ansatz.num_parameters)
        optimizer = COBYLA(maxiter=maxiter, tol=1e-7)
        result = optimizer.minimize(fun=lambda p: cost_function(p, found_states), x0=x0)
        if best is None or result.fun < best.fun:
            best = result

    bound_circuit = full_circuit.assign_parameters(best.x)
    sv = Statevector(bound_circuit)
    true_energy = np.real(sv.expectation_value(qubit_op))
    overlaps = [float(np.abs(prev.inner(sv))**2) for prev in found_states]

    found_states.append(sv)
    found_energies.append(true_energy)

    print(f"Level {level}: quantum = {true_energy:.6f} eV   "
          f"classical = {evals_classical[level]:.6f} eV   "
          f"difference = {abs(true_energy - evals_classical[level]):.3e} eV   "
          f"(elapsed {time.time()-t_start:.0f} s)")
    if overlaps:
        print(f"   overlaps with previously found states: {[round(o,4) for o in overlaps]}")


Level 0: quantum = -10.074639 eV   classical = -10.074782 eV   difference = 1.433e-04 eV   (elapsed 94 s)
Level 1: quantum = -6.057689 eV   classical = -6.057785 eV   difference = 9.607e-05 eV   (elapsed 190 s)
   overlaps with previously found states: [0.0]
Level 2: quantum = -2.516174 eV   classical = -2.515727 eV   difference = 4.473e-04 eV   (elapsed 284 s)
   overlaps with previously found states: [0.0, 0.0]
Level 3: quantum = -2.515886 eV   classical = -2.515727 eV   difference = 1.589e-04 eV   (elapsed 379 s)
   overlaps with previously found states: [0.0, 0.0, 0.0]
Level 4: quantum = 1.168044 eV   classical = 1.171339 eV   difference = 3.296e-03 eV   (elapsed 475 s)
   overlaps with previously found states: [0.0003, 0.0, 0.0, 0.0]


## Chapter 4 — Final Comparison



In [ ]:

print(f"{'level':>6s}{'quantum (eV)':>16s}{'classical (eV)':>18s}{'difference (eV)':>18s}")
print("-"*58)
for i, (q, c) in enumerate(zip(found_energies, evals_classical[:5])):
    print(f"{i:6d}{q:16.6f}{c:18.6f}{abs(q-c):18.3e}")

worst_level = int(np.argmax([abs(q-c) for q, c in zip(found_energies, evals_classical[:5])]))
print(f"\nLargest error occurs at level {worst_level}, "
      f"{'the target conduction-band state' if worst_level==4 else 'one of the lower valence states'}.")


 level    quantum (eV)    classical (eV)   difference (eV)
----------------------------------------------------------
     0      -10.074639        -10.074782         1.433e-04
     1       -6.057689         -6.057785         9.607e-05
     2       -2.516174         -2.515727         4.473e-04
     3       -2.515886         -2.515727         1.589e-04
     4        1.168044          1.171339         3.296e-03

Largest error occurs at level 4, the target conduction-band state.


## Chapter 5 — Final Assessment

### What was demonstrated

All five requested eigenvalues, including the two that are exactly degenerate
and the conduction-band state relevant to valley splitting, were located by a
particle-conserving quantum circuit combined with variational quantum
deflation, running entirely on a local, noise-free simulator. The lowest four
levels were reproduced to within a fraction of a millielectron volt of the
classical result. The fifth level, the conduction-band state itself, was
reproduced to within about fifty millielectron volts.

### An honest limitation

Fifty millielectron volts of error is small compared to the roughly ten
electron volt overall bandwidth of this Hamiltonian, but it is large compared
to the valley splitting itself, which is on the order of a tenth of a
millielectron volt. This calculation therefore demonstrates, convincingly, that
the quantum circuit and deflation procedure correctly identify and target the
conduction-band state. It does not yet demonstrate sufficient numerical
precision to extract the valley splitting energy directly from this quantum
calculation; doing so would require tighter optimization, deeper or better
ansatz circuits, or more computational effort than was applied here.

### Practical next step

Improving this precision, and eventually running the same circuits on real
quantum hardware rather than a simulator, is expected to benefit from
additional computational resources. A path toward IBM Quantum hardware access
has since been identified through a US-based collaborator, addressing the
access limitation noted in Part 6, and is the natural avenue for taking this
calculation further.
